<a href="https://colab.research.google.com/github/Aaricis/Hung-yi-Lee-ML2022/blob/main/HW7/ML2022Spring_HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 思路

## Simple Baseline(0.45139)
Score: 0.47882

Private score: 0.46569

跑一遍Sample Code。

## Medium Baseline(0.65792)

Score: 0.69302

Private score: 0.68684

- Apply linear learning rate decay;
  
  根据助教提示，使用带warm up 的 learning rate scheduler。

  ```python
  from transformers import get_linear_schedule_with_warmup

  # total training steps
  total_steps = len(train_loader) * num_epoch
  num_warmup_steps = int(0 * total_steps)  # Set warmup steps to 20% of total steps

  # [Hugging Face] Apply linear learning rate decay with warmup
  scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps
  )
  ```

- Change value of `doc_stride`;
  
  `doc_stride`表示两个连续窗口的起始位置之间的距离。默认值为`max_paragraph_len`，此时窗口是不重叠的，也就是说第一个窗口为[0, 149]，第二个窗口为[150, 299]......如果答案位于[140, 160]，默认的设置无法捕捉到答案。因此，需要调整`doc_stride`，使窗口之间发生重叠。`doc_stride`可以理解为截取文本时，窗口每次滑动的步长。

  `doc_stride`只在验证和测试阶段使用，与训练无关。经测试，`doc_stride`取`max_paragraph_len * 0.25`。
  ```python
  self.doc_stride = int(self.max_paragraph_len * 0.25)
  ```

## Strong Baseline(0.78136)
Score: 0.79548

Private score: 0.78974

- Improve preprocessing
  
  Sample Code以答案为中心截取训练文本，会让模型误以为答案都在文本的中心。我们增加随机偏移，让答案不总是在文本中心。

  ```python
  # 防止模型学习到「答案总是位于中间的位置」，加入随机偏移
  max_offset = self.max_paragraph_len // 2   # 最大偏移量为段落长度的1/2，这是可调的
  random_offset = np.random.randint(-max_offset, max_offset)  # 在 [-max_offset, +max_offset] 范围内随机选择偏移量
  paragraph_start = max(0, min(mid + random_offset - self.max_paragraph_len // 2, len(tokenized_paragraph) - self.max_paragraph_len))
  paragraph_end = paragraph_start + self.max_paragraph_len
  ```

- Try other pretrained models

  尝试不同的预训练模型，"hfl/chinese-roberta-wwm-ext-large"更强的RoBERTa中文WWM（Whole Word Masking，全词掩码模型）,表现好于Bert。
  
  >WWM（Whole Word Masking，全词掩码）是BERT及其变种模型中的一种预训练技术，主要用于改进中文（以及类似语言）的掩码语言建模（MLM）任务。它是针对原始BERT的字级别掩码（Character-level Masking）的优化方案。

  ```python
  model = BertForQuestionAnswering.from_pretrained("hfl/chinese-roberta-wwm-ext-large").to(device)
  tokenizer = BertTokenizerFast.from_pretrained("hfl/chinese-roberta-wwm-ext-large")
  ```




## Boss Baseline(0.84388)

Score: 0.84630

Private score: 0.83857

- Improve postprocessing

  Sample Code `def evaluate()`会出现`start_index`大于`end_index`的情况，导致`[start_index : end_index + 1]`无法捕获到答案。查看 result.csv 文件时，可以发现有些结果是空的，我们需要修正这个问题。
  - (1)只考虑`start_index < end_index`的区间，选择概率总和最大的区间。
  Score: 0.78660 Private score: 0.79217
  ```python
  def evaluate(data, output):
    answer = ''
    max_prob = float('-inf')
    num_of_windows = data[0].shape[1]

    for k in range(num_of_windows):
        start_logits = output.start_logits[k]  # shape: (seq_len,)
        end_logits = output.end_logits[k]     # shape: (seq_len,)
        
        # 向量化计算所有组合的概率和 (seq_len, seq_len)
        prob_matrix = start_logits.unsqueeze(1) + end_logits.unsqueeze(0)
        
        # 生成上三角掩码（确保end >= start）
        mask = torch.triu(torch.ones_like(prob_matrix, dtype=torch.bool))
        prob_matrix = prob_matrix.masked_fill(~mask, float('-inf'))
        
        # 找到最大概率的合法组合
        best_prob, best_idx = torch.max(prob_matrix.flatten(), dim=0)
        best_start, best_end = np.unravel_index(best_idx.item(), prob_matrix.shape)

        if best_prob > max_prob:
            max_prob = best_prob
            answer = tokenizer.decode(data[0][0][k][best_start : best_end + 1])

    return answer.replace(' ', '')
  ```
  - (2)直接跳过非法区间。
  
    Score: 0.78217  Private score: 0.78894
    ```python
    def evaluate(data, output):
    answer = ''
    max_prob = float('-inf')
    num_of_windows = data[0].shape[1]

    for k in range(num_of_windows):
        start_prob, start_index = torch.max(output.start_logits[k], dim=0)
        end_prob, end_index = torch.max(output.end_logits[k], dim=0)

        # 跳过非法区间
        if end_index < start_index:
            continue

        prob = start_prob + end_prob

        if prob > max_prob:
            max_prob = prob
            answer = tokenizer.decode(data[0][0][k][start_index : end_index + 1])

    return answer.replace(' ', '')
    ```
  - (3)枚举所有合法的区间，找出最大概率的那一对，并限制答案长度不超过`max_answer_length=30`。
  
  Score: 0.80072 Private score: 0.79782
  ```python
  def evaluate(data, output):
    answer = ''
    max_prob = float('-inf')
    num_of_windows = data[0].shape[1]
    max_answer_length = 30

    for k in range(num_of_windows):
        start_logits = output.start_logits[k]   # shape: (seq_len,)
        end_logits = output.end_logits[k]       # shape: (seq_len,)

        seq_len = start_logits.size(0)

        # 构造得分矩阵 score[i][j] = start_logits[i] + end_logits[j]
        start_logits = start_logits.unsqueeze(1)         # (seq_len, 1)
        end_logits = end_logits.unsqueeze(0)             # (1, seq_len)
        score_matrix = start_logits + end_logits         # (seq_len, seq_len)

        # 创建 mask：只保留满足 end >= start 且 长度 <= max_answer_length 的组合
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=0)  # end >= start
        mask = mask * torch.tril(torch.ones(seq_len, seq_len), diagonal=max_answer_length - 1)  # 限制长度
        mask = mask.to(score_matrix.device)

        # 将非法位置的分数设为极小值
        score_matrix = score_matrix.masked_fill(mask == 0, float('-inf'))

        # 找到最大得分及其索引
        flat_index = torch.argmax(score_matrix)
        start_index = flat_index // seq_len
        end_index = flat_index % seq_len

        prob = score_matrix[start_index, end_index]
        if prob > max_prob:
            max_prob = prob
            answer = tokenizer.decode(data[0][0][k][start_index:end_index + 1])

    return answer.replace(' ', '')
    ```

  综上，只有第三版`def evaluate()`Score略超过Strong Baseline，后续实验都使用此方法作推理。
- 梯度累积，即每n个step更新一次梯度，相当于将batchsize扩大n倍。
  使用梯度累积将batchsize扩大为64。
  ```python
  gradient_accumulation_steps = 2
  ```
  Score: 0.81202 Private score: 0.

- Adjust learning rate automatically by scheduler。
使用`get_cosine_schedule_with_warmup`，warmup step改为`1.15 * 0.1 * total_steps`。
```python
# total training steps
total_steps = len(train_loader) * num_epoch
num_warmup_steps= int(1.15 * 0.1 * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps)
```
- 其他
降低学习率到`1e-5`，增加epoch=4

- Ensemble多次尝试的结果。



# **Homework 7 - Bert (Question Answering)**

If you have any questions, feel free to email us at mlta-2022-spring@googlegroups.com



Slide:    [Link](https://docs.google.com/presentation/d/1H5ZONrb2LMOCixLY7D5_5-7LkIaXO6AGEaV2mRdTOMY/edit?usp=sharing)　Kaggle: [Link](https://www.kaggle.com/c/ml2022spring-hw7)　Data: [Link](https://drive.google.com/uc?id=1AVgZvy3VFeg0fX-6WQJMHPVrx3A-M1kb)




## Task description
- Chinese Extractive Question Answering
  - Input: Paragraph + Question
  - Output: Answer

- Objective: Learn how to fine tune a pretrained model on downstream task using transformers

- Todo
    - Fine tune a pretrained chinese BERT model
    - Change hyperparameters (e.g. doc_stride)
    - Apply linear learning rate decay
    - Try other pretrained models
    - Improve preprocessing
    - Improve postprocessing
- Training tips
    - Automatic mixed precision
    - Gradient accumulation
    - Ensemble

- Estimated training time (tesla t4 with automatic mixed precision enabled)
    - Simple: 8mins
    - Medium: 8mins
    - Strong: 25mins
    - Boss: 2.5hrs
  

## Download Dataset

In [ ]:
# Download link 1
!gdown --id '1AVgZvy3VFeg0fX-6WQJMHPVrx3A-M1kb' --output hw7_data.zip

# Download Link 2 (if the above link fails)
# !gdown --id '1qwjbRjq481lHsnTrrF4OjKQnxzgoLEFR' --output hw7_data.zip

# Download Link 3 (if the above link fails)
# !gdown --id '1QXuWjNRZH6DscSd6QcRER0cnxmpZvijn' --output hw7_data.zip

!unzip -o hw7_data.zip

# For this HW, K80 < P4 < T4 < P100 <= T4(fp16) < V100
!nvidia-smi

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1AVgZvy3VFeg0fX-6WQJMHPVrx3A-M1kb
To: /content/hw7_data.zip
100% 9.57M/9.57M [00:00<00:00, 179MB/s]
Archive:  hw7_data.zip
  inflating: hw7_dev.json            
  inflating: hw7_test.json           
  inflating: hw7_train.json          
Tue Apr 22 06:28:28 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. 

## Install transformers

Documentation for the toolkit:　https://huggingface.co/transformers/

In [ ]:
# You are allowed to change version of transformers or use other toolkits
# !pip install transformers==4.5.0
!pip install transformers

## Import Packages

In [ ]:
import json
import numpy as np
import random
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import BertForQuestionAnswering, BertTokenizerFast

from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

# Fix random seed for reproducibility
def same_seeds(seed):
	torch.manual_seed(seed)
	if torch.cuda.is_available():
		torch.cuda.manual_seed(seed)
		torch.cuda.manual_seed_all(seed)
	np.random.seed(seed)
	random.seed(seed)
	torch.backends.cudnn.benchmark = False
	torch.backends.cudnn.deterministic = True

same_seeds(0)

In [ ]:
# Change "fp16_training" to True to support automatic mixed precision training (fp16)
fp16_training = True

if fp16_training:
    # !pip install accelerate==0.2.0
    from accelerate import Accelerator
    # accelerator = Accelerator(fp16=True)
    accelerator = Accelerator(mixed_precision="fp16")
    device = accelerator.device

# Documentation for the toolkit:  https://huggingface.co/docs/accelerate/

## Load Model and Tokenizer






In [ ]:
! pip install huggingface-hub


In [ ]:
from huggingface_hub import hf_hub_download

# 指定清华镜像
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# model = BertForQuestionAnswering.from_pretrained("bert-base-chinese").to(device)
# tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")

# model = BertForQuestionAnswering.from_pretrained("hfl/chinese-bert-wwm-ext").to(device)
# tokenizer = BertTokenizerFast.from_pretrained("hfl/chinese-bert-wwm-ext")

# model = BertForQuestionAnswering.from_pretrained("hfl/chinese-roberta-wwm-ext").to(device)
# tokenizer = BertTokenizerFast.from_pretrained("hfl/chinese-roberta-wwm-ext")

# model = BertForQuestionAnswering.from_pretrained("hfl/chinese-roberta-wwm-ext-large").to(device)
# tokenizer = BertTokenizerFast.from_pretrained("hfl/chinese-roberta-wwm-ext-large")

model_name = "luhua/chinese_pretrain_mrc_macbert_large"
model = BertForQuestionAnswering.from_pretrained(model_name).to(device)
tokenizer = BertTokenizerFast.from_pretrained(model_name)

# You can safely ignore the warning message (it pops up because new prediction heads for QA are initialized randomly)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/669 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.30G [00:00<?, ?B/s]

Some weights of the model checkpoint at luhua/chinese_pretrain_mrc_macbert_large were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


model.safetensors:   0%|          | 0.00/1.30G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

## Read Data

- Training set: 31690 QA pairs
- Dev set: 4131  QA pairs
- Test set: 4957  QA pairs

- {train/dev/test}_questions:
  - List of dicts with the following keys:
   - id (int)
   - paragraph_id (int)
   - question_text (string)
   - answer_text (string)
   - answer_start (int)
   - answer_end (int)
- {train/dev/test}_paragraphs:
  - List of strings
  - paragraph_ids in questions correspond to indexs in paragraphs
  - A paragraph may be used by several questions

In [ ]:
def read_data(file):
    with open(file, 'r', encoding="utf-8") as reader:
        data = json.load(reader)
    return data["questions"], data["paragraphs"]

train_questions, train_paragraphs = read_data("hw7_train.json")
dev_questions, dev_paragraphs = read_data("hw7_dev.json")
test_questions, test_paragraphs = read_data("hw7_test.json")

## Tokenize Data

In [ ]:
# Tokenize questions and paragraphs separately
# 「add_special_tokens」 is set to False since special tokens will be added when tokenized questions and paragraphs are combined in datset __getitem__

train_questions_tokenized = tokenizer([train_question["question_text"] for train_question in train_questions], add_special_tokens=False)
dev_questions_tokenized = tokenizer([dev_question["question_text"] for dev_question in dev_questions], add_special_tokens=False)
test_questions_tokenized = tokenizer([test_question["question_text"] for test_question in test_questions], add_special_tokens=False)

train_paragraphs_tokenized = tokenizer(train_paragraphs, add_special_tokens=False)
dev_paragraphs_tokenized = tokenizer(dev_paragraphs, add_special_tokens=False)
test_paragraphs_tokenized = tokenizer(test_paragraphs, add_special_tokens=False)

# You can safely ignore the warning message as tokenized sequences will be futher processed in datset __getitem__ before passing to model

## Dataset and Dataloader

In [ ]:
class QA_Dataset(Dataset):
    def __init__(self, split, questions, tokenized_questions, tokenized_paragraphs):
        self.split = split
        self.questions = questions
        self.tokenized_questions = tokenized_questions
        self.tokenized_paragraphs = tokenized_paragraphs
        self.max_question_len = 40
        # self.max_paragraph_len = 150
        self.max_paragraph_len = 300

        ##### TODO: Change value of doc_stride #####
        self.doc_stride = 64
        # self.doc_stride = int(self.max_paragraph_len * 0.2)  # Set stride to 25% of max paragraph length (Medium)

        # Input sequence length = [CLS] + question + [SEP] + paragraph + [SEP]
        self.max_seq_len = 1 + self.max_question_len + 1 + self.max_paragraph_len + 1

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        question = self.questions[idx]
        tokenized_question = self.tokenized_questions[idx]
        tokenized_paragraph = self.tokenized_paragraphs[question["paragraph_id"]]

        ##### TODO: Preprocessing #####
        # Hint: How to prevent model from learning something it should not learn

        if self.split == "train":
            # Convert answer's start/end positions in paragraph_text to start/end positions in tokenized_paragraph
            answer_start_token = tokenized_paragraph.char_to_token(question["answer_start"])
            answer_end_token = tokenized_paragraph.char_to_token(question["answer_end"])

            # A single window is obtained by slicing the portion of paragraph containing the answer
            mid = (answer_start_token + answer_end_token) // 2
            # 防止模型学习到「答案总是位于中间的位置」，加入随机偏移
            max_offset = self.max_paragraph_len // 2   # 最大偏移量为段落长度的1/2，这是可调的
            random_offset = np.random.randint(-max_offset, max_offset)  # 在 [-max_offset, +max_offset] 范围内随机选择偏移量
            # paragraph_start = max(0, min(mid - self.max_paragraph_len // 2, len(tokenized_paragraph) - self.max_paragraph_len))
            paragraph_start = max(0, min(mid + random_offset - self.max_paragraph_len // 2, len(tokenized_paragraph) - self.max_paragraph_len))
            paragraph_end = paragraph_start + self.max_paragraph_len

            # Slice question/paragraph and add special tokens (101: CLS, 102: SEP)
            input_ids_question = [101] + tokenized_question.ids[:self.max_question_len] + [102]
            input_ids_paragraph = tokenized_paragraph.ids[paragraph_start : paragraph_end] + [102]

            # Convert answer's start/end positions in tokenized_paragraph to start/end positions in the window
            answer_start_token += len(input_ids_question) - paragraph_start
            answer_end_token += len(input_ids_question) - paragraph_start

            # Pad sequence and obtain inputs to model
            input_ids, token_type_ids, attention_mask = self.padding(input_ids_question, input_ids_paragraph)
            return torch.tensor(input_ids), torch.tensor(token_type_ids), torch.tensor(attention_mask), answer_start_token, answer_end_token

        # Validation/Testing
        else:
            input_ids_list, token_type_ids_list, attention_mask_list = [], [], []

            # Paragraph is split into several windows, each with start positions separated by step "doc_stride"
            for i in range(0, len(tokenized_paragraph), self.doc_stride):

                # Slice question/paragraph and add special tokens (101: CLS, 102: SEP)
                input_ids_question = [101] + tokenized_question.ids[:self.max_question_len] + [102]
                input_ids_paragraph = tokenized_paragraph.ids[i : i + self.max_paragraph_len] + [102]

                # Pad sequence and obtain inputs to model
                input_ids, token_type_ids, attention_mask = self.padding(input_ids_question, input_ids_paragraph)

                input_ids_list.append(input_ids)
                token_type_ids_list.append(token_type_ids)
                attention_mask_list.append(attention_mask)

            return torch.tensor(input_ids_list), torch.tensor(token_type_ids_list), torch.tensor(attention_mask_list)

    def padding(self, input_ids_question, input_ids_paragraph):
        # Pad zeros if sequence length is shorter than max_seq_len
        padding_len = self.max_seq_len - len(input_ids_question) - len(input_ids_paragraph)
        # Indices of input sequence tokens in the vocabulary
        input_ids = input_ids_question + input_ids_paragraph + [0] * padding_len
        # Segment token indices to indicate first and second portions of the inputs. Indices are selected in [0, 1]
        token_type_ids = [0] * len(input_ids_question) + [1] * len(input_ids_paragraph) + [0] * padding_len
        # Mask to avoid performing attention on padding token indices. Mask values selected in [0, 1]
        attention_mask = [1] * (len(input_ids_question) + len(input_ids_paragraph)) + [0] * padding_len

        return input_ids, token_type_ids, attention_mask

train_set = QA_Dataset("train", train_questions, train_questions_tokenized, train_paragraphs_tokenized)
# dev_set = QA_Dataset("dev", dev_questions, dev_questions_tokenized, dev_paragraphs_tokenized)
dev_set = QA_Dataset("train", dev_questions, dev_questions_tokenized, dev_paragraphs_tokenized)
test_set = QA_Dataset("test", test_questions, test_questions_tokenized, test_paragraphs_tokenized)

# train_batch_size = 32
train_batch_size = 8

# Note: Do NOT change batch size of dev_loader / test_loader !
# Although batch size=1, it is actually a batch consisting of several windows from the same QA pair

from torch.utils.data import ConcatDataset
combined_train_set = ConcatDataset([train_set, dev_set])
train_loader = DataLoader(combined_train_set, batch_size=train_batch_size, shuffle=True, pin_memory=True)

# train_loader = DataLoader(train_set, batch_size=train_batch_size, shuffle=True, pin_memory=True)
dev_loader = DataLoader(dev_set, batch_size=1, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=1, shuffle=False, pin_memory=True)

## Function for Evaluation

In [ ]:
# def evaluate(data, output):
#     ##### TODO: Postprocessing #####
#     # There is a bug and room for improvement in postprocessing
#     # Hint: Open your prediction file to see what is wrong

#     answer = ''
#     max_prob = float('-inf')
#     num_of_windows = data[0].shape[1]

#     for k in range(num_of_windows):
#         # Obtain answer by choosing the most probable start position / end position
#         start_prob, start_index = torch.max(output.start_logits[k], dim=0)
#         end_prob, end_index = torch.max(output.end_logits[k], dim=0)

#         # Probability of answer is calculated as sum of start_prob and end_prob
#         prob = start_prob + end_prob

#         # Replace answer if calculated probability is larger than previous windows
#         if prob > max_prob:
#             max_prob = prob
#             # Convert tokens to chars (e.g. [1920, 7032] --> "大 金")
#             answer = tokenizer.decode(data[0][0][k][start_index : end_index + 1])

#     # Remove spaces in answer (e.g. "大 金" --> "大金")
#     return answer.replace(' ','')

In [ ]:
# def evaluate(data, output):
#     answer = ''
#     max_prob = float('-inf')
#     num_of_windows = data[0].shape[1]

#     for k in range(num_of_windows):
#         start_logits = output.start_logits[k]  # shape: (seq_len,)
#         end_logits = output.end_logits[k]     # shape: (seq_len,)

#         # 向量化计算所有组合的概率和 (seq_len, seq_len)
#         prob_matrix = start_logits.unsqueeze(1) + end_logits.unsqueeze(0)

#         # 生成上三角掩码（确保end >= start）
#         mask = torch.triu(torch.ones_like(prob_matrix, dtype=torch.bool))
#         prob_matrix = prob_matrix.masked_fill(~mask, float('-inf'))

#         # 找到最大概率的合法组合
#         best_prob, best_idx = torch.max(prob_matrix.flatten(), dim=0)
#         best_start, best_end = np.unravel_index(best_idx.item(), prob_matrix.shape)

#         if best_prob > max_prob:
#             max_prob = best_prob
#             answer = tokenizer.decode(data[0][0][k][best_start : best_end + 1])

#     return answer.replace(' ', '')

In [ ]:
# def evaluate(data, output):
#     answer = ''
#     max_prob = float('-inf')
#     num_of_windows = data[0].shape[1]

#     for k in range(num_of_windows):
#         start_prob, start_index = torch.max(output.start_logits[k], dim=0)
#         end_prob, end_index = torch.max(output.end_logits[k], dim=0)

#         # 跳过非法区间
#         if end_index < start_index:
#             continue

#         prob = start_prob + end_prob

#         if prob > max_prob:
#             max_prob = prob
#             answer = tokenizer.decode(data[0][0][k][start_index : end_index + 1])

#     return answer.replace(' ', '')


In [ ]:
# def evaluate(data, output, paragraph=None, paragraph_tokenized=None):
#     answer = ''
#     max_prob = float('-inf')
#     num_of_windows = data[0].shape[1]
#     max_answer_length = 30

#     for k in range(num_of_windows):
#         start_logits = output.start_logits[k]   # shape: (seq_len,)
#         end_logits = output.end_logits[k]       # shape: (seq_len,)

#         seq_len = start_logits.size(0)

#         # 构造得分矩阵 score[i][j] = start_logits[i] + end_logits[j]
#         start_logits = start_logits.unsqueeze(1)         # (seq_len, 1)
#         end_logits = end_logits.unsqueeze(0)             # (1, seq_len)
#         score_matrix = start_logits + end_logits         # (seq_len, seq_len)

#         # 创建 mask：只保留满足 end >= start 且 长度 <= max_answer_length 的组合
#         mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=0)  # end >= start
#         mask = mask * torch.tril(torch.ones(seq_len, seq_len), diagonal=max_answer_length - 1)  # 限制长度
#         mask = mask.to(score_matrix.device)

#         # 将非法位置的分数设为极小值
#         score_matrix = score_matrix.masked_fill(mask == 0, float('-inf'))

#         # 找到最大得分及其索引
#         flat_index = torch.argmax(score_matrix)
#         start_index = flat_index // seq_len
#         end_index = flat_index % seq_len

#         prob = score_matrix[start_index, end_index]
#         if prob > max_prob:
#             max_prob = prob
#             answer = tokenizer.decode(data[0][0][k][start_index:end_index + 1])

#     answer = answer.replace(' ', '')
#     if '[UNK]' in answer:
#         try:
#             # Use original paragraph text if there are any unknown tokens
#             raw_start = paragraph_tokenized.token_to_chars(start_index)[0]
#             raw_end = paragraph_tokenized.token_to_chars(end_index)[1]
#             answer = paragraph[raw_start: raw_end]
#         except:
#             print("NoneType Error, just ignore for now!")

#     return answer


In [ ]:
def evaluate(data, output, doc_stride=64, token_type_ids=None, paragraph=None, paragraph_tokenized=None):
    ##### TODO: Postprocessing #####
    # There is a bug and room for improvement in postprocessing
    # Hint: Open your prediction file to see what is wrong

    answer = ''
    max_prob = float('-inf')
    num_of_windows = data[0].shape[1]

    for k in range(num_of_windows):
        # Obtain answer by choosing the most probable start position / end position
        start_prob, start_index = torch.max(output.start_logits[k], dim=0)
        end_prob, end_index = torch.max(output.end_logits[k], dim=0)

        token_type_id = data[1][0][k].detach().cpu().numpy()
        #[CLS] + [question] + [SEP] + [paragraph] + [SEP]
        paragraph_start = token_type_id.argmax()
        paragraph_end = len(token_type_id) - 1 - token_type_id[::-1].argmax()-1

        if start_index > end_index or start_index < paragraph_start or end_index > paragraph_end:
            continue

        # Probability of answer is calculated as sum of start_prob and end_prob
        prob = start_prob + end_prob

        # Replace answer if calculated probability is larger than previous windows
        if prob > max_prob:
            # Convert tokens to chars (e.g. [1920, 7032] --> "大 金")
            max_prob = prob
            answer = tokenizer.decode(data[0][0][k][start_index : end_index + 1])
            # 找到tokenized paragraph中对应的位置
            origin_start = start_index + k * doc_stride - paragraph_start
            origin_end = end_index + k * doc_stride - paragraph_start;

    answer = answer.replace(' ', '')
    if '[UNK]' in answer:
        print('发现 [UNK]，这表明有文字无法编码, 使用原始文本')
        #print("Paragraph:", paragraph)
        #print("Paragraph:", paragraph_tokenized.tokens)
        print('--直接解码预测:', answer)
        #找到原始文本中对应的位置
        raw_start =  paragraph_tokenized.token_to_chars(origin_start)[0]
        raw_end = paragraph_tokenized.token_to_chars(origin_end)[1]
        answer = paragraph[raw_start:raw_end]
        print('--原始文本预测:',answer)

    # Remove spaces in answer (e.g. "大 金" --> "大金")
    return answer

## Training

In [ ]:
num_epoch = 2
validation = False
logging_step = 100
learning_rate = 1e-5 #1e-4

optimizer = AdamW(model.parameters(), lr=learning_rate)

#---- Medium -----
# from transformers import get_linear_schedule_with_warmup
from transformers import get_cosine_schedule_with_warmup

# total training steps
total_steps = len(train_loader) * num_epoch
# num_warmup_steps = int(0.2 * total_steps)  # Set warmup steps to 20% of total steps
num_warmup_steps= int(1.15 * 0.1 * total_steps)

# [Hugging Face] Apply linear learning rate decay with warmup
# scheduler = get_linear_schedule_with_warmup(
#     optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps
# )
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps)


if fp16_training:
    model, optimizer, train_loader, scheduler = accelerator.prepare(model, optimizer, train_loader, scheduler)

model.train()

print("Start Training ...")

for epoch in range(num_epoch):
    step = 1
    train_loss = train_acc = 0

    for data in tqdm(train_loader):
        # Load all data into GPU
        data = [i.to(device) for i in data]

        # Model inputs: input_ids, token_type_ids, attention_mask, start_positions, end_positions (Note: only "input_ids" is mandatory)
        # Model outputs: start_logits, end_logits, loss (return when start_positions/end_positions are provided)
        output = model(input_ids=data[0], token_type_ids=data[1], attention_mask=data[2], start_positions=data[3], end_positions=data[4])

        # Choose the most probable start position / end position
        start_index = torch.argmax(output.start_logits, dim=1)
        end_index = torch.argmax(output.end_logits, dim=1)

        # Prediction is correct only if both start_index and end_index are correct
        train_acc += ((start_index == data[3]) & (end_index == data[4])).float().mean()
        train_loss += output.loss

        if fp16_training:
            accelerator.backward(output.loss)
        else:
            output.loss.backward()

        optimizer.step()
        optimizer.zero_grad()
        step += 1

        ##### TODO: Apply linear learning rate decay #####
        scheduler.step()

        # Print training loss and accuracy over past logging step
        if step % logging_step == 0:
            print(f"Epoch {epoch + 1} | Step {step} | loss = {train_loss.item() / logging_step:.3f}, acc = {train_acc / logging_step:.3f}")
            train_loss = train_acc = 0

    if validation:
        print("Evaluating Dev Set ...")
        model.eval()
        with torch.no_grad():
            dev_acc = 0
            for i, data in enumerate(tqdm(dev_loader)):
                output = model(input_ids=data[0].squeeze(dim=0).to(device), token_type_ids=data[1].squeeze(dim=0).to(device),
                       attention_mask=data[2].squeeze(dim=0).to(device))
                # prediction is correct only if answer text exactly matches
                dev_acc += evaluate(data, output) == dev_questions[i]["answer_text"]
            print(f"Validation | Epoch {epoch + 1} | acc = {dev_acc / len(dev_loader):.3f}")
        model.train()

# Save a model and its configuration file to the directory 「saved_model」
# i.e. there are two files under the direcory 「saved_model」: 「pytorch_model.bin」 and 「config.json」
# Saved model can be re-loaded using 「model = BertForQuestionAnswering.from_pretrained("saved_model")」
print("Saving Model ...")
model_save_dir = "saved_model"
model.save_pretrained(model_save_dir)

Start Training ...


  0%|          | 0/4478 [00:00<?, ?it/s]

Epoch 1 | Step 100 | loss = 0.085, acc = 0.949
Epoch 1 | Step 200 | loss = 0.083, acc = 0.961
Epoch 1 | Step 300 | loss = 0.101, acc = 0.952
Epoch 1 | Step 400 | loss = 0.071, acc = 0.959
Epoch 1 | Step 500 | loss = 0.098, acc = 0.959
Epoch 1 | Step 600 | loss = 0.086, acc = 0.957
Epoch 1 | Step 700 | loss = 0.088, acc = 0.944
Epoch 1 | Step 800 | loss = 0.130, acc = 0.936
Epoch 1 | Step 900 | loss = 0.084, acc = 0.951
Epoch 1 | Step 1000 | loss = 0.146, acc = 0.938
Epoch 1 | Step 1100 | loss = 0.147, acc = 0.927
Epoch 1 | Step 1200 | loss = 0.115, acc = 0.942
Epoch 1 | Step 1300 | loss = 0.148, acc = 0.925
Epoch 1 | Step 1400 | loss = 0.155, acc = 0.936
Epoch 1 | Step 1500 | loss = 0.140, acc = 0.926
Epoch 1 | Step 1600 | loss = 0.157, acc = 0.924
Epoch 1 | Step 1700 | loss = 0.136, acc = 0.922
Epoch 1 | Step 1800 | loss = 0.165, acc = 0.919
Epoch 1 | Step 1900 | loss = 0.130, acc = 0.939
Epoch 1 | Step 2000 | loss = 0.167, acc = 0.911
Epoch 1 | Step 2100 | loss = 0.136, acc = 0.944
E

  0%|          | 0/4478 [00:00<?, ?it/s]

Epoch 2 | Step 100 | loss = 0.088, acc = 0.957
Epoch 2 | Step 200 | loss = 0.051, acc = 0.969
Epoch 2 | Step 300 | loss = 0.053, acc = 0.970
Epoch 2 | Step 400 | loss = 0.040, acc = 0.979
Epoch 2 | Step 500 | loss = 0.055, acc = 0.971
Epoch 2 | Step 600 | loss = 0.045, acc = 0.976
Epoch 2 | Step 700 | loss = 0.039, acc = 0.976
Epoch 2 | Step 800 | loss = 0.067, acc = 0.970
Epoch 2 | Step 900 | loss = 0.065, acc = 0.969
Epoch 2 | Step 1000 | loss = 0.052, acc = 0.972
Epoch 2 | Step 1100 | loss = 0.058, acc = 0.966
Epoch 2 | Step 1200 | loss = 0.052, acc = 0.972
Epoch 2 | Step 1300 | loss = 0.047, acc = 0.979
Epoch 2 | Step 1400 | loss = 0.068, acc = 0.974
Epoch 2 | Step 1500 | loss = 0.068, acc = 0.965
Epoch 2 | Step 1600 | loss = 0.044, acc = 0.977
Epoch 2 | Step 1700 | loss = 0.048, acc = 0.972
Epoch 2 | Step 1800 | loss = 0.048, acc = 0.970
Epoch 2 | Step 1900 | loss = 0.050, acc = 0.965
Epoch 2 | Step 2000 | loss = 0.042, acc = 0.976
Epoch 2 | Step 2100 | loss = 0.070, acc = 0.974
E

In [ ]:
# num_epoch = 2
# validation = False
# logging_step = 100
# learning_rate = 1e-4
# gradient_accumulation_steps = 2

# optimizer = AdamW(model.parameters(), lr=learning_rate)

# from transformers import get_linear_schedule_with_warmup

# total_steps = len(train_loader) * num_epoch // gradient_accumulation_steps
# num_warmup_steps = int(0.2 * total_steps)

# scheduler = get_linear_schedule_with_warmup(
#     optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps
# )

# if fp16_training:
#     model, optimizer, train_loader, scheduler = accelerator.prepare(model, optimizer, train_loader, scheduler)

# model.train()
# print("Start Training ...")

# for epoch in range(num_epoch):
#     step = 0
#     train_loss = 0.0
#     train_acc = 0.0
#     acc_step = 0

#     for data in tqdm(train_loader):
#         data = [i.to(device) for i in data]

#         output = model(input_ids=data[0], token_type_ids=data[1], attention_mask=data[2], start_positions=data[3], end_positions=data[4])

#         start_index = torch.argmax(output.start_logits, dim=1)
#         end_index = torch.argmax(output.end_logits, dim=1)

#         batch_acc = ((start_index == data[3]) & (end_index == data[4])).float().mean().item()
#         train_acc += batch_acc
#         acc_step += 1

#         loss = output.loss / gradient_accumulation_steps
#         train_loss += loss.item()

#         if fp16_training:
#             accelerator.backward(loss)
#         else:
#             loss.backward()

#         if (step + 1) % gradient_accumulation_steps == 0:
#             optimizer.step()
#             scheduler.step()
#             optimizer.zero_grad()

#             if (step + 1) % (gradient_accumulation_steps * logging_step) == 0:
#                 print(f"Epoch {epoch + 1} | Step {(step + 1) // gradient_accumulation_steps} | loss = {train_loss:.3f}, acc = {train_acc / acc_step:.3f}")
#                 train_loss = 0.0
#                 train_acc = 0.0
#                 acc_step = 0

#         step += 1

#     # if validation:
#     #     print("Evaluating Dev Set ...")
#     #     model.eval()
#     #     with torch.no_grad():
#     #         dev_acc = 0
#     #         for i, data in enumerate(tqdm(dev_loader)):
#     #             output = model(input_ids=data[0].squeeze(dim=0).to(device), token_type_ids=data[1].squeeze(dim=0).to(device),
#     #                            attention_mask=data[2].squeeze(dim=0).to(device))
#     #             dev_acc += evaluate(data, output) == dev_questions[i]["answer_text"]
#     #         print(f"Validation | Epoch {epoch + 1} | acc = {dev_acc / len(dev_loader):.3f}")
#     #     model.train()

# print("Saving Model ...")
# model_save_dir = "saved_model"
# model.save_pretrained(model_save_dir)


Start Training ...


  0%|          | 0/1120 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 10.12 MiB is free. Process 22291 has 14.73 GiB memory in use. Of the allocated memory 14.46 GiB is allocated by PyTorch, and 148.99 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Testing

In [ ]:
print("Evaluating Test Set ...")

result = []

model.eval()
with torch.no_grad():
    # for data in tqdm(test_loader):
    for i, data in enumerate(tqdm(test_loader)):
        output = model(input_ids=data[0].squeeze(dim=0).to(device), token_type_ids=data[1].squeeze(dim=0).to(device),
                       attention_mask=data[2].squeeze(dim=0).to(device))
        # result.append(evaluate(
        #     data, output,
        #     paragraph=test_paragraphs[test_questions[i]["paragraph_id"]],
        #     paragraph_tokenized=test_paragraphs_tokenized[test_questions[i]["paragraph_id"]]))
        result.append(evaluate(data, output, doc_stride=64, paragraph=test_paragraphs[test_questions[i]["paragraph_id"]],
                               paragraph_tokenized=test_paragraphs_tokenized[test_questions[i]["paragraph_id"]]))

result_file = "result.csv"
with open(result_file, 'w') as f:
  f.write("ID,Answer\n")
  for i, test_question in enumerate(test_questions):
    # Replace commas in answers with empty strings (since csv is separated by comma)
    # Answers in kaggle are processed in the same way
    f.write(f"{test_question['id']},{result[i].replace(',','')}\n")

print(f"Completed! Result is in {result_file}")

Evaluating Test Set ...


  0%|          | 0/4957 [00:00<?, ?it/s]

NoneType Error, just ignore for now!
Completed! Result is in result.csv


## Ensemble_多数投票法（Majority Voting）

In [14]:
import pandas as pd
from collections import Counter
import glob

# 读取所有结果 CSV 文件
csv_paths = glob.glob('result*.csv')  # 或者手动指定 ['result1.csv', 'result2.csv', 'result3.csv']
results = [pd.read_csv(path) for path in csv_paths]

# 假设所有 CSV 文件中 ID 顺序一致
ids = results[0]['ID']
answers_by_id = {}

# 聚集每个 ID 的答案
for idx in ids:
    answers = [df.loc[df['ID'] == idx, 'Answer'].values[0] for df in results]
    vote = Counter(answers).most_common(1)[0][0]
    answers_by_id[idx] = vote

# 保存ensemble结果
ensemble_df = pd.DataFrame({'ID': ids, 'Answer': [answers_by_id[idx] for idx in ids]})
ensemble_df.to_csv('ensemble_result.csv', index=False)


## Ensemble_优先非[CLS] + 投票

In [15]:
def majority_vote_with_cls_fallback(answers):
    non_cls = [a for a in answers if a != '[CLS]']
    if non_cls:
        return Counter(non_cls).most_common(1)[0][0]
    return '[CLS]'

# 替换 Counter 逻辑为上面这个函数即可

import pandas as pd
from collections import Counter
import glob

# 读取所有结果 CSV 文件
csv_paths = glob.glob('result*.csv')  # 或者手动指定 ['result1.csv', 'result2.csv', 'result3.csv']
results = [pd.read_csv(path) for path in csv_paths]

# 假设所有 CSV 文件中 ID 顺序一致
ids = results[0]['ID']
answers_by_id = {}

# 聚集每个 ID 的答案
for idx in ids:
    answers = [df.loc[df['ID'] == idx, 'Answer'].values[0] for df in results]
    # vote = Counter(answers).most_common(1)[0][0]
    vote = majority_vote_with_cls_fallback(answers)
    answers_by_id[idx] = vote

# 保存ensemble结果
ensemble_df = pd.DataFrame({'ID': ids, 'Answer': [answers_by_id[idx] for idx in ids]})
ensemble_df.to_csv('ensemble_result_wo_cls.csv', index=False)
